In [1]:
import os
import re
import json
import numpy as np
import pandas as pd
import torch
import time
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from sklearn.metrics import f1_score, classification_report

import warnings
warnings.filterwarnings('ignore')

DATA_DIR  = "./Data"
MODEL_DIR = "./Models_CE"

dev_df = pd.read_csv(os.path.join(DATA_DIR, "dev.csv"))
print(f"Dev: {len(dev_df):,}")

start = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()
print(f"Model load: {time.time()-start:.1f}s")
print(f"Device: {device}")

with open(os.path.join(MODEL_DIR, "best_threshold.json")) as f:
    best_threshold = json.load(f)['best_threshold']
print(f"Threshold: {best_threshold:.2f}")

def clean_text(text):
    text = str(text)
    text = re.sub(r'(From|To|Cc|Subject|Date|Forwarded by)[^\n]*\n', '', text)
    text = re.sub(r'\S+@\S+\.\S+', '', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

MAX_LENGTH = 512
BATCH_SIZE = 32

class AVDataset(Dataset):
    def __init__(self, df):
        df = df.copy()
        df['text_1'] = df['text_1'].apply(clean_text)
        df['text_2'] = df['text_2'].apply(clean_text)
        self.encodings = tokenizer(
            list(df['text_1']),
            list(df['text_2']),
            max_length=MAX_LENGTH,
            truncation='longest_first',
            padding=False,
            return_tensors=None
        )

    def __len__(self):
        return len(self.encodings['input_ids'])

    def __getitem__(self, idx):
        return {
            'input_ids': torch.tensor(self.encodings['input_ids'][idx]),
            'attention_mask': torch.tensor(self.encodings['attention_mask'][idx]),
        }

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print("✅ Ready!")

Dev: 5,993
Model load: 2.2s
Device: cuda
Threshold: 0.38
✅ Ready!


In [2]:
start = time.time()
dev_dataset = AVDataset(dev_df)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=0,
                        collate_fn=data_collator)
print(f"Tokenizing: {time.time()-start:.1f}s")

start = time.time()
all_probs = []
with torch.no_grad():
    for batch in dev_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=-1)[:, 1]
        all_probs.extend(probs.cpu().numpy())

print(f"Inference: {time.time()-start:.1f}s")
final_preds = (np.array(all_probs) >= best_threshold).astype(int)

print(f"\nF1 Score: {f1_score(dev_df['label'], final_preds, average='macro'):.4f}")
print(classification_report(dev_df['label'], final_preds))

Tokenizing: 1.2s
Inference: 353.9s

F1 Score: 0.8410
              precision    recall  f1-score   support

           0       0.82      0.86      0.84      2937
           1       0.86      0.82      0.84      3056

    accuracy                           0.84      5993
   macro avg       0.84      0.84      0.84      5993
weighted avg       0.84      0.84      0.84      5993



In [3]:
demo_df = pd.read_csv(os.path.join(DATA_DIR, "dev.csv"))
print(f"Demo: {len(demo_df):,}")

start = time.time()
demo_dataset = AVDataset(demo_df)
demo_loader = DataLoader(demo_dataset, batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=0,
                         collate_fn=data_collator)
print(f"Tokenizing: {time.time()-start:.1f}s")

start = time.time()
all_probs = []
with torch.no_grad():
    for batch in demo_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=-1)[:, 1]
        all_probs.extend(probs.cpu().numpy())

print(f"Inference: {time.time()-start:.1f}s")
final_preds = (np.array(all_probs) >= best_threshold).astype(int)

pd.DataFrame({'prediction': final_preds}).to_csv("GroupN_AV_C_Prediction.csv", index=False)
print("✅ Saved → GroupN_AV_C_Prediction.csv")

Demo: 5,993
Inference: 420.0s
✅ Saved → GroupN_AV_C_Prediction.csv
